# Assignment 1 — manual checks

Private study aid, not a required submission file. Select the project's `.venv` Python kernel, restart it, and run the cells from top to bottom. No packages are installed here.

Read each expected result before running its cell. An `assert` stops the cell when the actual result differs. The small DNA strings below are deliberately constructed examples, not real genome data. To experiment, change a toy input and work out its expected answer by hand.


In [2]:
import csv
import hashlib
import math
import subprocess
import sys
import tempfile
from pathlib import Path

assignment_dir = Path.cwd()
if not (assignment_dir / 'genome.fasta').is_file():
    assignment_dir = assignment_dir / 'assignment_1'
assert (assignment_dir / 'genome.fasta').is_file(), 'Open this notebook from the project root or assignment_1.'

sys.path.insert(0, str(assignment_dir))
from Bio.Seq import Seq
from orf_finder import load_genome, find_orfs
from kmer_freq import count_kmers

original_hashes = {}
for filename in ('genome.fasta', 'orfs.csv', 'kmers_k3.csv'):
    original_hashes[filename] = hashlib.sha256((assignment_dir / filename).read_bytes()).hexdigest()

print('Python kernel:', sys.executable)
print('Assignment directory:', assignment_dir)


Python kernel: d:\Studiowe\DataScience\Sem 1\Bioinformatics\Projekt 1\.venv\Scripts\python.exe
Assignment directory: d:\Studiowe\DataScience\Sem 1\Bioinformatics\Projekt 1\assignment_1


## 1. Read the real genome

Expected: lambda phage `NC_001416.1`, **48,502 nucleotides**, using only uppercase A, C, G and T. The loader uses Biopython to read exactly one FASTA record.


In [4]:
genome = load_genome(assignment_dir / 'genome.fasta')
print('Genome:', genome.id)
print('Length:', len(genome.seq))
print('First 30 bases:', genome.seq[:30])
assert genome.id == 'NC_001416.1'
assert len(genome.seq) == 48502
assert set(str(genome.seq)) <= set('ACGT')
print('Genome loading passed.')


Genome: NC_001416.1
Length: 48502
First 30 bases: GGGCGGCGACCTCGCGGGTTTTCGCTATTT
Genome loading passed.


## 2. Check all six reading frames

`ATG AAA TAA` translates to **MK**. Its length is **9 nt**, including STOP, but its protein omits the terminal `*`. Prefixing one or two bases shifts the forward frame.

Expected rows below use **1-based inclusive coordinates on the original forward reference**. In the three reverse examples, the ORF occupies positions 1–9; trailing forward-reference bases shift the reverse-complement frame. A negative frame gives the orientation, while start stays below end.


In [6]:
dna = Seq('ATGAAATAA')

expected = {
    'frame': '+1',
    'start': 1,
    'end': 9,
    'length': 9,
    'protein': 'MK',
}

actual = find_orfs(dna)
print('Expected:', expected)
print('Actual:', actual)

assert actual == [expected], 'ORF check failed.'
print('Check passed.')

Expected: {'frame': '+1', 'start': 1, 'end': 9, 'length': 9, 'protein': 'MK'}
Actual: [{'frame': '+1', 'start': 1, 'end': 9, 'length': 9, 'protein': 'MK'}]
Check passed.


## 3. Check starts and stops

For `ATG ATG TAA TAG`, both ATGs start a candidate. Expected proteins are **MM** and **M**, both ending at position **9** (the first in-frame STOP), rather than at the later TAG. `ATGAAA` has no STOP and must produce no complete ORF.


In [7]:
nested_orfs = find_orfs(Seq('ATGATGTAATAG'))
expected_orfs = [
    {'frame': '+1', 'start': 1, 'end': 9, 'length': 9, 'protein': 'MM'},
    {'frame': '+1', 'start': 4, 'end': 9, 'length': 6, 'protein': 'M'},
]
print('Nested starts:', nested_orfs)
assert nested_orfs == expected_orfs
assert find_orfs(Seq('ATGAAA')) == []
print('Nested starts, first STOP and unfinished ORF checks passed.')


Nested starts: [{'frame': '+1', 'start': 1, 'end': 9, 'length': 9, 'protein': 'MM'}, {'frame': '+1', 'start': 4, 'end': 9, 'length': 6, 'protein': 'M'}]
Nested starts, first STOP and unfinished ORF checks passed.


## 4. Count k-mers by hand

For `AAACAA`, the five overlapping 2-mers are **AA, AA, AC, CA, AA**. Expected: AA = 3/5, AC = 1/5, CA = 1/5. Counts decrease, with alphabetical ordering for ties. `AAAA` has three overlapping AA windows, not two. Only the supplied forward sequence is counted.


In [8]:
expected_kmers = [('AA', 3, 0.6), ('AC', 1, 0.2), ('CA', 1, 0.2)]
actual_kmers = count_kmers(Seq('AAACAA'), 2)
for row in actual_kmers:
    print(row)
assert actual_kmers == expected_kmers
assert count_kmers(Seq('AAAA'), 2) == [('AA', 3, 1.0)]
assert count_kmers(Seq('ATG'), 3) == [('ATG', 1, 1.0)]
print('Counts, frequencies, overlap, ordering and forward-only checks passed.')


('AA', 3, 0.6)
('AC', 1, 0.2)
('CA', 1, 0.2)
Counts, frequencies, overlap, ordering and forward-only checks passed.


## 5. Run the actual programs and check their CSVs

Both commands use the notebook kernel's Python. All new outputs go to a temporary directory that is removed afterwards; the saved CSVs are never used as output paths.

Expected for the included genome: **708 ORFs** with a minimum of 100 nt, and **64 distinct 3-mers** covering **48,500 windows**. Regenerated files must match the saved CSVs byte for byte. A separate 9-nt toy ORF must be included at `--min-len 9` and excluded at `--min-len 10`, verifying the inclusive threshold.


In [9]:
with tempfile.TemporaryDirectory() as temporary:
    temporary_dir = Path(temporary)
    commands = [
        ['orf_finder.py', 'genome.fasta', '--min-len', '100', '--out', str(temporary_dir / 'orfs.csv')],
        ['kmer_freq.py', 'genome.fasta', '-k', '3', '--out', str(temporary_dir / 'kmers_k3.csv')],
    ]
    for arguments in commands:
        result = subprocess.run([sys.executable, '-B'] + arguments, cwd=assignment_dir,
                                capture_output=True, text=True)
        print(result.stdout.strip())
        assert result.returncode == 0, result.stderr
    for filename in ('orfs.csv', 'kmers_k3.csv'):
        assert (temporary_dir / filename).read_bytes() == (assignment_dir / filename).read_bytes()
        print(filename, 'matches the saved file exactly.')

    toy_fasta = temporary_dir / 'toy.fasta'
    toy_fasta.write_text('>toy\nATGAAATAA\n', encoding='utf-8')
    for minimum, expected_count in [(9, 1), (10, 0)]:
        toy_csv = temporary_dir / 'toy.csv'
        arguments = [sys.executable, '-B', 'orf_finder.py', str(toy_fasta),
                     '--min-len', str(minimum), '--out', str(toy_csv)]
        result = subprocess.run(arguments, cwd=assignment_dir, capture_output=True, text=True)
        assert result.returncode == 0, result.stderr
        with toy_csv.open(newline='', encoding='utf-8') as handle:
            rows = list(csv.DictReader(handle))
        assert len(rows) == expected_count
        print('Minimum length:', minimum, 'ORFs:', len(rows))


Wrote 708 ORFs to C:\Users\misie\AppData\Local\Temp\tmpo8v4oztd\orfs.csv.
Wrote 64 distinct 3-mers to C:\Users\misie\AppData\Local\Temp\tmpo8v4oztd\kmers_k3.csv.
orfs.csv matches the saved file exactly.
kmers_k3.csv matches the saved file exactly.
Minimum length: 9 ORFs: 1
Minimum length: 10 ORFs: 0


In [10]:
with (assignment_dir / 'orfs.csv').open(newline='', encoding='utf-8') as handle:
    saved_orfs = list(csv.DictReader(handle))
with (assignment_dir / 'kmers_k3.csv').open(newline='', encoding='utf-8') as handle:
    saved_kmers = list(csv.DictReader(handle))

assert len(saved_orfs) == 708
for orf in saved_orfs:
    length = int(orf['length'])
    assert length >= 100
    assert int(orf['end']) - int(orf['start']) + 1 == length
    assert len(orf['protein']) == length // 3 - 1

window_count = 0
frequencies = []
for row in saved_kmers:
    window_count += int(row['count'])
    frequencies.append(float(row['frequency']))
assert len(saved_kmers) == 64
assert window_count == len(genome.seq) - 3 + 1 == 48500
assert math.isclose(math.fsum(frequencies), 1.0, rel_tol=0, abs_tol=1e-12)
print('ORFs:', len(saved_orfs), 'Distinct 3-mers:', len(saved_kmers), 'Windows:', window_count)
print('Frequency sum:', math.fsum(frequencies))

for filename, original_hash in original_hashes.items():
    current_hash = hashlib.sha256((assignment_dir / filename).read_bytes()).hexdigest()
    assert current_hash == original_hash, filename + ' changed during the checks.'
print('All checks passed. The original genome and CSV files are unchanged.')


ORFs: 708 Distinct 3-mers: 64 Windows: 48500
Frequency sum: 1.0
All checks passed. The original genome and CSV files are unchanged.


## What these checks establish

The toy cases provide independently hand-counted expectations. CSV comparisons check that the saved results are reproducible with the current scripts; totals and row properties check their consistency. These checks are not proof for every possible input, and sequence-defined ORFs are not confirmed genes. Restart the kernel before repeating the notebook after editing a script so Python loads the updated functions.
